# SIEVE — Phase 2 : monde ouvert (*open-world*) et robustesse

**Même projet que le notebook 1** (`SIEVE_Reproduction_Extension.ipynb`), même dossier `./data`, même chargeur et même prétraitement. Ce notebook ajoute **deux volets complémentaires** :

| Volet | Question | Où |
|---|---|---|
| **A. Open-world** | Un classifieur de logs peut-il *reconnaître qu'il ne connaît pas* une classe (nouveau type de log) au lieu de la ranger de force dans une classe connue ? | §3-5 |
| **B. Robustesse** | Deux solutions améliorent-elles la tenue des modèles face au **bruit / à l'entropie** de SPICE *et* la détection de l'inconnu ? | §6-7 |
| **C. Deep learning** (optionnel) | Un LSTM se comporte-t-il mieux qu'un modèle linéaire sur la détection de l'inconnu ? | §8 |

## Protocole open-world (6 plis de 5 classes)
Les 30 classes sont réparties en **6 groupes disjoints de 5 classes**. À chaque pli :
1. les **25 classes connues** servent à l'entraînement, à la validation et au test (*closed-set*) ;
2. les **5 classes inconnues** ne sont **jamais vues** à l'entraînement ni à la validation : elles servent uniquement à tester la détection de l'inconnu ;
3. on remet les 5 classes, on en retire 5 autres, et on recommence : **6 fois**, de sorte que chaque classe est inconnue exactement une fois.

Ce découpage **par classes entières** est plus propre que le protocole `grouped` de la phase 1 : il n'y a aucune fuite de gabarit entre classes connues et inconnues, par construction.

### Deux façons de former les plis
- **`random`** : tirage aléatoire (graine fixe) : c'est ton protocole.
- **`family`** : les classes d'une même famille (`file-*`, `process-*`, `user-*`…) sont retirées *ensemble*. C'est le mode **difficile** : si `file-deleted` est inconnue alors que `directory-deleted` (quasi identique) est connue, le modèle a de bonnes raisons de se tromper. On compare les deux modes pour séparer « le détecteur est mauvais » de « la classe inconnue ressemble trop à une classe connue ».

## Métriques
- **Closed-set** (25 classes connues) : accuracy, F1 macro.
- **Détection de l'inconnu** (positif = inconnu) : **AUROC**, **AUPR**, et **FPR@95 %TPR** (part d'inconnus acceptés à tort quand on accepte 95 % des connus). Plus c'est bas, mieux c'est.
- **Seuil de rejet τ** fixé sur la **validation des classes connues seulement** (95 % des connus acceptés) : jamais avec les inconnues.
- **Open-set F1 macro** à 26 sorties (25 connues + « inconnu »).
- Moyennes ± écart-type **sur les 6 plis**.

## Les deux solutions de robustesse testées
- **S1 — entraînement multi-entropie + bruit** : mélanger les niveaux d'entropie de SPICE (SIEVE_00, 01, 02…) et perturber 50 % des logs d'entraînement (échange et suppression de tokens), **à budget d'entraînement constant** (même nombre de logs que la référence : le gain ne vient donc pas d'un surplus de données).
- **S2 — features robustes + calibration** : TF-IDF **mots + caractères** (n-grammes de caractères, moins sensibles aux synonymes et aux fautes) puis **calibration par température** (les scores deviennent de vraies confiances, ce qui sert directement la détection de l'inconnu).
- **S1+S2** : les deux ensemble.

Ce sont des **hypothèses à tester**, pas des résultats acquis : le notebook mesure leur effet réel.

## 0. Configuration (identique au notebook 1, plus les réglages de la phase 2)
`QUICK_MODE=True` (ou variable d'environnement `SIEVE_QUICK=1`) réduit tout pour un essai de quelques minutes. Les réglages propres à cette phase sont ajoutés dans la cellule suivante.

In [ ]:
from __future__ import annotations

import copy
import json
import os
import platform
import random
import re
import sys
import time
import warnings
from collections import Counter
from functools import lru_cache
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)
sns.set_theme(style="whitegrid", context="notebook")

# ------------------------------------------------------------------ CONFIG
CFG = SimpleNamespace(
    # --- chemins (relatifs : le projet reste portable)
    DATA_DIR="./data",
    OUT_DIR="./outputs",
    SEED=42,
    QUICK_MODE=os.environ.get("SIEVE_QUICK", "0") == "1",   # mettre True pour un test rapide

    # --- lecture des données (None = détection automatique, à vérifier au §1)
    TEXT_COL=None,            # nom de la colonne contenant le log brut
    LABEL_COL=None,           # nom de la colonne contenant la classe
    HEADER=None,              # None=auto | True | False  (CSV/TSV)
    LABEL_FROM="auto",        # fichiers .log/.txt : "auto" | "filename" | "parent_folder"
    LINE_SEPARATOR=None,      # fichiers .log/.txt de type "label<SEP>log" (ex. "\t" ou ",")
    LINE_LABEL_POSITION="first",   # "first" ou "last" (position du label dans la ligne)
    REAL_DATASET_REGEX=r"cds|real|combined",   # nom du dataset de LOGS RÉELS (test externe)
    ACTIVE_DATASET=None,      # nom du dataset d'entraînement (None = le plus grand non réel)
    MAX_ROWS_PER_DATASET=None,

    # --- prétraitement
    DROP_DUPLICATES=True,     # supprime les doublons exacts (log, label)
    DROP_NUMBERS=False,       # l'article conserve les nombres (ex. "2808")

    # --- protocoles
    TEST_SIZE=0.25,           # 75/25 comme l'article
    PROTOCOLS=None,           # None = tous ceux disponibles

    # --- modèles classiques
    TUNE=False,               # True = petite GridSearch sur C (LR / LinearSVC)
    SVM_RBF_MAX_TRAIN=15000,  # SVC RBF (celui de l'article) : sous-échantillon car O(n²)

    # --- RNN / LSTM (PyTorch)
    RNN_MAX_TRAIN=None, RNN_EPOCHS=6, RNN_BATCH=256, RNN_LR=2e-3,
    EMB_DIM=50, HIDDEN=128, LAYERS=2, RNN_MAX_LEN=64,

    # --- Transformers (GPU 4 Go : batch 16 x accumulation 2 = batch effectif 32)
    BERT_CHECKPOINT="bert-base-uncased",
    DISTIL_CHECKPOINT="distilbert-base-uncased",
    TF_MAX_TRAIN=30000, TF_EPOCHS=2, TF_BATCH=16, TF_ACCUM=2, TF_LR=2e-5, TF_MAX_LEN_CAP=128,

    # --- extensions
    ENTROPY_MAX_TRAIN=30000,
)

if CFG.QUICK_MODE:                       # exécution de contrôle
    CFG.MAX_ROWS_PER_DATASET = 4000
    CFG.SVM_RBF_MAX_TRAIN = 1500
    CFG.RNN_EPOCHS = 2
    CFG.TF_MAX_TRAIN, CFG.TF_EPOCHS, CFG.TF_BATCH, CFG.TF_ACCUM = 600, 1, 8, 1
    CFG.ENTROPY_MAX_TRAIN = 3000

OUT = Path(CFG.OUT_DIR)
(OUT / "figures").mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------- PyTorch (optionnel)
try:
    import torch
    HAS_TORCH = True
except Exception as exc:                 # ImportError ou DLL manquante (Windows)
    torch, HAS_TORCH = None, False
    print(f"[info] PyTorch indisponible ({type(exc).__name__}) : les sections RNN/LSTM/BERT/DistilBERT seront ignorées.")

try:
    import transformers  # noqa: F401
    HAS_TRANSFORMERS = True
except Exception:
    HAS_TRANSFORMERS = False


def set_seed(seed: int = CFG.SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    if HAS_TORCH:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)


def get_device() -> str:
    if HAS_TORCH and torch.cuda.is_available():
        return "cuda"
    if HAS_TORCH and getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


set_seed()
DEVICE = get_device()

print(f"Python {sys.version.split()[0]} | {platform.system()} | seed={CFG.SEED} | QUICK_MODE={CFG.QUICK_MODE}")
print(f"pandas {pd.__version__} | numpy {np.__version__}")
if HAS_TORCH:
    print(f"torch {torch.__version__} | device = {DEVICE}")
    if DEVICE == "cuda":
        props = torch.cuda.get_device_properties(0)
        print(f"GPU : {props.name} | VRAM : {props.total_memory / 1e9:.1f} Go")
print(f"transformers disponible : {HAS_TRANSFORMERS}")

In [ ]:
# ---------------- Réglages propres à la phase 2 ----------------
CFG.N_FOLDS = 6                 # 6 plis ...
CFG.UNKNOWN_PER_FOLD = 5        # ... de 5 classes inconnues (30 classes au total)
CFG.FOLD_MODES = ["random", "family"]
CFG.MAX_PER_CLASS = 2000        # logs par classe conservés pour le volet A (vitesse)
CFG.MAX_PER_CLASS_ROBUST = 800  # logs par classe et par dataset pour le volet B
CFG.MAX_EVAL_PER_CLASS = 500    # plafond de logs d'évaluation par classe (inconnues / dataset décalé)
CFG.SPLIT = (0.60, 0.15, 0.25)  # train / validation / test des classes CONNUES
CFG.MODELS_A = ["LogisticRegression", "LinearSVC"]
CFG.DETECTORS_A = ["MSP", "MaxLogit", "Energy", "Mahalanobis", "kNN"]
CFG.DETECTORS_B = ["MSP", "Energy"]
CFG.B_MODEL = "LinearSVC"       # modèle des volets B ("LinearSVC" rapide ou "LogisticRegression")
CFG.B_FOLDS = None              # None = les 6 plis ; ex. [1, 2] pour aller plus vite
CFG.RUN_DEEP = True             # volet C (nécessite PyTorch)
CFG.DEEP_EPOCHS = 3
CFG.DEEP_MAX_TRAIN = 30000
CFG.DEEP_FOLDS = None

if CFG.QUICK_MODE:
    CFG.MAX_PER_CLASS, CFG.MAX_PER_CLASS_ROBUST, CFG.MAX_EVAL_PER_CLASS = 150, 100, 60
    CFG.DEEP_EPOCHS, CFG.DEEP_MAX_TRAIN = 1, 3000
print("Phase 2 : plis =", CFG.N_FOLDS, "x", CFG.UNKNOWN_PER_FOLD, "classes inconnues | modes :", CFG.FOLD_MODES)

## 1. Données : découverte, chargement, vérification
Cellules **identiques au notebook 1** (déjà éprouvées), notamment le chargeur qui découpe chaque ligne au *premier* séparateur : les logs de SIEVE contiennent des virgules non protégées et un parseur CSV standard en perdrait près de 19 %.

In [ ]:
DATA_DIR = Path(CFG.DATA_DIR)
assert DATA_DIR.exists(), (
    f"Dossier introuvable : {DATA_DIR.resolve()}\n"
    "Crée un dossier 'data' à côté du notebook et place-y les fichiers de logs."
)

ALL_FILES = sorted(p for p in DATA_DIR.rglob("*") if p.is_file() and not p.name.startswith("."))
assert ALL_FILES, f"Aucun fichier dans {DATA_DIR.resolve()}"


def count_lines(path: Path, limit_mb: float = 300) -> int | None:
    if path.stat().st_size > limit_mb * 1e6:
        return None
    with path.open("rb") as f:
        return sum(1 for _ in f)


inventory = pd.DataFrame(
    {
        "fichier": [str(p.relative_to(DATA_DIR)) for p in ALL_FILES],
        "extension": [p.suffix.lower() or "(aucune)" for p in ALL_FILES],
        "taille_Mo": [round(p.stat().st_size / 1e6, 3) for p in ALL_FILES],
        "lignes": [count_lines(p) for p in ALL_FILES],
    }
)
display(inventory)

ARCHIVE_EXT = {".zip", ".gz", ".tar", ".tgz", ".7z", ".rar"}
archives = [p for p in ALL_FILES if p.suffix.lower() in ARCHIVE_EXT]
if archives:
    print("\n[ATTENTION] Archives détectées (non lues) : décompresse-les dans ./data :", [a.name for a in archives])

print("\n--- Aperçu brut (3 premières lignes des 12 premiers fichiers texte) ---")
for p in [p for p in ALL_FILES if p.suffix.lower() not in ARCHIVE_EXT | {".parquet", ".xlsx", ".xls"}][:12]:
    print(f"\n# {p.relative_to(DATA_DIR)}")
    with p.open("r", encoding="utf-8", errors="replace") as f:
        for _ in range(3):
            print("  ", repr(f.readline()[:220]))

In [ ]:
TABULAR_EXT = {".csv", ".tsv", ".json", ".jsonl", ".parquet", ".xlsx", ".xls"}
LINE_EXT = {".log", ".txt", ".dat", ""}
TEXT_HINT = re.compile(r"log|text|message|msg|content|raw|event|line|entry", re.I)
LABEL_HINT = re.compile(r"label|class|categor|type|target|action|^y$", re.I)


def sniff_sep(path: Path, n: int = 4096) -> str:
    import csv
    with path.open("r", encoding="utf-8", errors="replace") as f:
        head = f.read(n)
    first = head.splitlines()[0] if head else ""
    for cand in [",", "\t", ";", "|"]:                  # en-tête à exactement 2 colonnes ?
        if cand in first and len(first.split(cand)) == 2:
            return cand
    try:
        return csv.Sniffer().sniff(head, delimiters=",;\t|").delimiter
    except csv.Error:
        return "\t" if head.count("\t") > head.count(",") else ","


def detect_header(path: Path, sep: str) -> bool:
    if CFG.HEADER is not None:
        return bool(CFG.HEADER)
    with path.open("r", encoding="utf-8", errors="replace") as f:
        first = f.readline().rstrip("\r\n")
    cells = [c.strip().strip('"') for c in first.split(sep)]
    short = [c for c in cells if len(c) < 40]
    return bool(short) and max(len(c) for c in cells) < 60 and any(
        TEXT_HINT.search(c) or LABEL_HINT.search(c) for c in short
    )


def read_table(path: Path) -> pd.DataFrame:
    ext = path.suffix.lower()
    if ext in {".csv", ".tsv"}:
        sep = "\t" if ext == ".tsv" else sniff_sep(path)
        has_head = detect_header(path, sep)
        header = 0 if has_head else None
        # Cas fréquent pour des logs : fichier « label,log » où le log contient lui-même des virgules NON protégées.
        # Un parseur CSV standard perdrait ces lignes (ou les couperait en plusieurs colonnes) -> on découpe
        # chaque ligne au PREMIER séparateur seulement.
        if has_head:
            with path.open("r", encoding="utf-8", errors="replace") as f:
                head_cells = f.readline().rstrip("\r\n").split(sep)
            if len(head_cells) == 2:
                rows, no_sep = [], 0
                with path.open("r", encoding="utf-8", errors="replace", newline="") as f:
                    f.readline()
                    for ln in f:
                        ln = ln.rstrip("\r\n")
                        if not ln.strip():
                            continue
                        i = ln.find(sep)
                        if i < 0:
                            no_sep += 1
                        else:
                            rows.append((ln[:i], ln[i + 1:]))
                print(f"[info] {path.name} : lu en mode 2 colonnes (découpage au premier « {sep!r} ») -> "
                      f"{len(rows)} lignes ; {no_sep} ligne(s) sans séparateur ignorée(s)")
                return pd.DataFrame(rows, columns=[c.strip().strip('"') for c in head_cells])
        try:
            df = pd.read_csv(path, sep=sep, header=header, dtype=str, encoding_errors="replace", on_bad_lines="warn")
        except Exception:
            df = pd.read_csv(path, sep=sep, header=header, dtype=str, encoding_errors="replace",
                             engine="python", on_bad_lines="skip")
        expected = (count_lines(path) or 0) - (1 if has_head else 0)
        if expected and len(df) < 0.999 * expected:
            print(f"[ATTENTION] {path.name} : {expected - len(df)} ligne(s) perdue(s) par le parseur CSV "
                  f"({len(df)} lues sur {expected}). Vérifie le format (virgules ou guillemets dans les logs).")
        if header is None:
            df.columns = [f"col_{i}" for i in range(df.shape[1])]
        return df
    if ext == ".jsonl":
        return pd.read_json(path, lines=True, dtype=False)
    if ext == ".json":
        try:
            return pd.read_json(path, dtype=False)
        except ValueError:
            return pd.read_json(path, lines=True, dtype=False)
    if ext == ".parquet":
        return pd.read_parquet(path)
    return pd.read_excel(path, dtype=str)


def guess_columns(df: pd.DataFrame) -> tuple[str, str | None]:
    cols = list(df.columns)
    mean_len = {c: df[c].astype(str).str.len().mean() for c in cols}
    nuniq = {c: df[c].nunique(dropna=True) for c in cols}
    text_col = CFG.TEXT_COL or max(cols, key=lambda c: mean_len[c] + (50 if TEXT_HINT.search(str(c)) else 0))
    label_col = CFG.LABEL_COL
    if label_col is None:
        cand = [c for c in cols if c != text_col and 2 <= nuniq[c] <= min(500, max(50, 0.05 * len(df))) and nuniq[c] < len(df)]
        if cand:
            label_col = sorted(cand, key=lambda c: (not LABEL_HINT.search(str(c)), nuniq[c]))[0]
    return text_col, label_col


def infer_label_mode(files: list[Path]) -> str:
    """Pour les fichiers .log/.txt : le label est-il le nom du fichier ou le dossier parent ?"""
    line_files = [p for p in files if p.suffix.lower() in LINE_EXT]
    rels = [p.relative_to(DATA_DIR).parts for p in line_files]
    if not rels or all(len(r) == 1 for r in rels):
        return "filename"
    if any(len(r) >= 3 for r in rels):
        return "parent_folder"                       # data/<dataset>/<classe>/<fichier>
    folders_of_stem: dict[str, set] = {}
    for r in rels:
        if len(r) == 2:
            folders_of_stem.setdefault(Path(r[1]).stem, set()).add(r[0])
    repeated = any(len(v) >= 2 for v in folders_of_stem.values())
    if not repeated:
        return "parent_folder"
    # mêmes noms de fichiers dans plusieurs dossiers : l'axe le plus riche (classes) porte le label,
    # l'autre (peu de valeurs) désigne les datasets
    n_folders = len({r[0] for r in rels if len(r) == 2})
    return "filename" if len(folders_of_stem) > n_folders else "parent_folder"


LINE_MODE = infer_label_mode(ALL_FILES)
print(f"Fichiers .log/.txt : label déduit du « {LINE_MODE} » (modifiable via CFG.LABEL_FROM)")


def read_lines_file(path: Path) -> pd.DataFrame:
    rel = path.relative_to(DATA_DIR)
    parts = rel.parts
    with path.open("r", encoding="utf-8", errors="replace") as f:
        lines = [ln.rstrip("\r\n") for ln in f if ln.strip()]
    if CFG.LINE_SEPARATOR:
        sep = CFG.LINE_SEPARATOR
        split = [ln.split(sep, 1) if CFG.LINE_LABEL_POSITION == "first" else ln.rsplit(sep, 1) for ln in lines]
        ok = [s for s in split if len(s) == 2]
        if CFG.LINE_LABEL_POSITION == "first":
            labels, texts = zip(*ok)
        else:
            texts, labels = zip(*ok)
        return pd.DataFrame({"text": texts, "label": labels, "source_file": str(rel)}), path.stem
    mode = CFG.LABEL_FROM if CFG.LABEL_FROM != "auto" else LINE_MODE
    if len(parts) == 1:
        mode = "filename"
    label = parts[-2] if mode == "parent_folder" and len(parts) >= 2 else path.stem
    if mode == "parent_folder":
        dataset = parts[-3] if len(parts) >= 3 else DATA_DIR.name
    else:
        dataset = parts[-2] if len(parts) >= 2 else DATA_DIR.name
    return pd.DataFrame({"text": lines, "label": label, "source_file": str(rel)}), dataset


def load_all() -> tuple[dict[str, pd.DataFrame], pd.DataFrame]:
    parts: dict[str, list[pd.DataFrame]] = {}
    summary = []
    for p in ALL_FILES:
        ext = p.suffix.lower()
        if ext in ARCHIVE_EXT:
            continue
        rel = str(p.relative_to(DATA_DIR))
        if ext in TABULAR_EXT:
            raw = read_table(p)
            if raw.shape[1] < 2 and not CFG.LINE_SEPARATOR:
                print(f"[skip] {rel} : une seule colonne ; indique TEXT_COL/LABEL_COL ou LINE_SEPARATOR.")
                continue
            tcol, lcol = guess_columns(raw)
            if lcol is None:
                print(f"[skip] {rel} : aucune colonne label plausible. Renseigne CFG.LABEL_COL (colonnes : {list(raw.columns)}).")
                continue
            part = pd.DataFrame({"text": raw[tcol], "label": raw[lcol], "source_file": rel})
            name, used = p.stem, f"text={tcol!r}, label={lcol!r}"
        elif ext in LINE_EXT:
            part, name = read_lines_file(p)
            used = f"label={'ligne' if CFG.LINE_SEPARATOR else CFG.LABEL_FROM}"
        else:
            print(f"[skip] {rel} : extension non gérée ({ext}).")
            continue
        parts.setdefault(name, []).append(part)
        summary.append({"fichier": rel, "dataset": name, "lignes": len(part), "colonnes_utilisees": used})
    datasets = {k: pd.concat(v, ignore_index=True) for k, v in parts.items()}
    return datasets, pd.DataFrame(summary)


def sample_per_class(df: pd.DataFrame, n_total: int, label_col: str = "label") -> pd.DataFrame:
    if n_total is None or len(df) <= n_total:
        return df
    return df.groupby(label_col, group_keys=False, sort=False).sample(frac=n_total / len(df), random_state=CFG.SEED)


DATASETS, load_summary = load_all()
assert DATASETS, "Aucun dataset chargé : voir les messages [skip] ci-dessus et ajuste CFG."
display(load_summary)

if CFG.MAX_ROWS_PER_DATASET:
    DATASETS = {k: sample_per_class(v, CFG.MAX_ROWS_PER_DATASET) for k, v in DATASETS.items()}

print("\nDatasets chargés :")
for k, v in DATASETS.items():
    print(f"  - {k:<25} {len(v):>8} logs | {v['label'].nunique():>3} classes")

In [ ]:
real_candidates = [k for k in DATASETS if re.search(CFG.REAL_DATASET_REGEX, k, re.I)]
synthetic = [k for k in DATASETS if k not in real_candidates]

if CFG.ACTIVE_DATASET:
    ACTIVE_NAME = CFG.ACTIVE_DATASET
elif synthetic:
    ACTIVE_NAME = max(synthetic, key=lambda k: len(DATASETS[k]))
else:
    ACTIVE_NAME = max(DATASETS, key=lambda k: len(DATASETS[k]))
    real_candidates = []
REAL_NAME = next((k for k in real_candidates if k != ACTIVE_NAME), None)

print(f"Dataset ACTIF (train/test)  : {ACTIVE_NAME}")
print(f"Dataset RÉEL (test externe) : {REAL_NAME}")
print(f"Autres datasets (extension entropie) : {[k for k in DATASETS if k not in (ACTIVE_NAME, REAL_NAME)]}")

### 1.1 Audit et nettoyage minimal
Même audit qu'en phase 1 : valeurs manquantes, doublons, conflits de labels, déséquilibre. Aucun modèle n'est entraîné avant ce contrôle.

In [ ]:
def normalize_label(s: pd.Series) -> pd.Series:
    return (s.astype(str).str.strip().str.lower()
              .str.replace(r"[\s_]+", "-", regex=True))


def audit(name: str, df: pd.DataFrame) -> dict:
    d = df.copy()
    d["label_n"] = normalize_label(d["label"].where(d["label"].notna(), "")).replace("", np.nan)
    text = d["text"].astype("string")
    stats = {
        "dataset": name,
        "lignes": len(d),
        "texte_manquant": int(text.isna().sum()),
        "texte_vide": int((text.fillna("").str.strip() == "").sum()),
        "label_manquant": int(d["label_n"].isna().sum()),
        "doublons_exacts(log,label)": int(d.duplicated(subset=["text", "label_n"]).sum()),
        "conflits(log->>1 label)": int((d.dropna(subset=["label_n"]).groupby("text")["label_n"].nunique() > 1).sum()),
        "n_classes": int(d["label_n"].nunique()),
    }
    vc = d["label_n"].value_counts()
    stats["ratio_desequilibre(max/min)"] = round(vc.max() / max(vc.min(), 1), 2) if len(vc) else np.nan
    return stats


audit_df = pd.DataFrame([audit(k, v) for k, v in DATASETS.items()]).set_index("dataset")
display(audit_df.T)

raw_active = DATASETS[ACTIVE_NAME]
print(f"\n=== Structure du dataset actif : {ACTIVE_NAME} ===")
raw_active.info()
display(raw_active.head(8))

In [ ]:
def clean_frame(raw: pd.DataFrame, drop_dups: bool = CFG.DROP_DUPLICATES) -> pd.DataFrame:
    d = raw[["text", "label", "source_file"]].copy()
    d["text"] = d["text"].astype("string").str.strip()
    d["label"] = normalize_label(d["label"].where(d["label"].notna(), ""))
    n0 = len(d)
    d = d[(d["text"].notna()) & (d["text"] != "") & (d["label"] != "") & (d["label"] != "nan")]
    n1 = len(d)
    if drop_dups:
        d = d.drop_duplicates(subset=["text", "label"])
    d["text"] = d["text"].astype(str)
    print(f"  nettoyage : {n0} -> {n1} (valeurs manquantes/vides) -> {len(d)} (doublons exacts)")
    return d.reset_index(drop=True)


print("Nettoyage minimal du dataset ACTIF :")
ACTIVE_RAW = clean_frame(raw_active)
REAL_RAW = None
if REAL_NAME:
    print("Nettoyage minimal du dataset RÉEL :")
    REAL_RAW = clean_frame(DATASETS[REAL_NAME], drop_dups=False)   # on garde tout le test réel

n_classes = ACTIVE_RAW["label"].nunique()
assert n_classes >= 2, (
    f"Une seule classe détectée ({ACTIVE_RAW['label'].unique()}). "
    "Vérifie CFG.LABEL_COL / CFG.LABEL_FROM / CFG.LINE_SEPARATOR."
)
print(f"\nClasses du dataset actif : {n_classes}")

if REAL_RAW is not None:
    only_real = sorted(set(REAL_RAW["label"]) - set(ACTIVE_RAW["label"]))
    only_syn = sorted(set(ACTIVE_RAW["label"]) - set(REAL_RAW["label"]))
    print(f"Classes présentes UNIQUEMENT dans le réel : {only_real}")
    print(f"Classes présentes UNIQUEMENT dans le synthétique : {only_syn}")
    if only_real:
        print("  -> si ces noms sont des variantes d'écriture, harmonise les labels avant de continuer.")

In [ ]:
EDA_DONE = {"flag": False}

vc = ACTIVE_RAW["label"].value_counts()
checks = {
    "au moins 2 classes": ACTIVE_RAW["label"].nunique() >= 2,
    "pas de valeurs manquantes": not ACTIVE_RAW[["text", "label"]].isna().any().any(),
    "pas de texte vide": bool((ACTIVE_RAW["text"].str.strip() != "").all()),
    "assez de classes pour le protocole": ACTIVE_RAW["label"].nunique() >= CFG.UNKNOWN_PER_FOLD * 2,
}
for k, ok in checks.items():
    print(("[OK] " if ok else "[ÉCHEC] ") + k)
assert all(checks.values()), "Données insuffisantes pour le protocole open-world."
print(f"\n{ACTIVE_RAW['label'].nunique()} classes | min={vc.min()} max={vc.max()} logs par classe (ratio {vc.max() / vc.min():.2f})")
EDA_DONE["flag"] = True


def require_eda() -> None:
    if not EDA_DONE["flag"]:
        raise RuntimeError("Exécute d'abord la vérification des données (section 1).")

## 2. Prétraitement et encodage
Même pipeline qu'en phase 1 (suppression IP/URL/dates/chemins/ponctuation, mots grammaticaux sans les négations, stemming Snowball). Les perturbations de la section 6 sont appliquées **sur les tokens déjà nettoyés** (plus rapide, et équivalent à perturber le texte puis le nettoyer, aux détails de ponctuation près).

In [ ]:
STOPWORDS = set(
    "a about above after again against all am an and any are as at be because been before being below between both "
    "but by can could did do does doing during each few for from further had has have having he her here hers him his "
    "how i if in into is it its itself just me more most my myself of on once only or other our over own same she "
    "should so some such than that the their theirs them then there these they this those through to too under until "
    "very was we were what when where which while who whom why will with would you your yours".split()
)   # volontairement SANS : no, not, nor, never, cannot, up, down, out, off (porteurs de sens en logs)

In [ ]:
from nltk.stem.snowball import SnowballStemmer

_MONTH = (r"(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|june?|july?|aug(?:ust)?|"
          r"sep(?:t(?:ember)?)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)")
RE_URL = re.compile(r"(?:https?|ftp)://\S+|www\.\S+", re.I)
RE_IPV4 = re.compile(r"\b\d{1,3}(?:\.\d{1,3}){3}(?::\d+)?\b")
RE_IPV6 = re.compile(r"\b(?:[0-9a-f]{1,4}:){2,7}[0-9a-f]{1,4}\b", re.I)
RE_DATE = re.compile(rf"\b{_MONTH}\s+\d{{1,2}}(?:,?\s+\d{{4}})?\b|\b\d{{4}}-\d{{2}}-\d{{2}}\b|\b\d{{1,2}}/\d{{1,2}}/\d{{2,4}}\b", re.I)
RE_TIME = re.compile(r"\b\d{1,2}:\d{2}(?::\d{2})?(?:[.,]\d+)?\b")
RE_PATH = re.compile(r"(?:[A-Za-z]:\\|~?/)[^\s:;,'\"]+")
RE_PUNCT = re.compile(r"[^\w\s]|_")

STEMMER = SnowballStemmer("english")


@lru_cache(maxsize=200_000)
def stem(word: str) -> str:
    return STEMMER.stem(word)


def preprocess(text: str) -> str:
    """Pipeline de nettoyage de l'article (IP/URL -> dates/heures -> chemins/ponctuation -> stopwords -> stemming)."""
    t = RE_URL.sub(" ", text)
    t = RE_IPV4.sub(" ", t)
    t = RE_IPV6.sub(" ", t)
    t = RE_DATE.sub(" ", t)
    t = RE_TIME.sub(" ", t)
    t = RE_PATH.sub(" ", t)
    t = RE_PUNCT.sub(" ", t).lower()
    tokens = [w for w in t.split() if w not in STOPWORDS]
    if CFG.DROP_NUMBERS:
        tokens = [w for w in tokens if not w.isdigit()]
    tokens = [stem(w) for w in tokens]
    return " ".join(tokens) if tokens else "emptylog"


# Démonstration étape par étape (comme le Tableau 3 de l'article) sur 3 logs du dataset actif
demo = ACTIVE_RAW["text"].sample(3, random_state=CFG.SEED).tolist()
for s in demo:
    print("BRUT   :", s)
    print("NETTOYÉ:", preprocess(s), "\n")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder().fit(sorted(ACTIVE_RAW["label"].unique()))
CLASSES = list(le.classes_)
K = len(CLASSES)
print(f"{K} classes :", CLASSES)


def sample_per_class(df: pd.DataFrame, n: int | None, seed: int = CFG.SEED) -> pd.DataFrame:
    """Au plus n logs par classe (tirage reproductible)."""
    if n is None:
        return df.reset_index(drop=True)
    rng = np.random.default_rng(seed)
    idx = []
    for _, ix in df.groupby("label").indices.items():
        ix = np.asarray(ix)
        idx.append(rng.choice(ix, size=min(n, len(ix)), replace=False))
    return df.iloc[np.sort(np.concatenate(idx))].reset_index(drop=True)


def prep_frame(raw: pd.DataFrame, max_per_class: int | None) -> pd.DataFrame:
    d = sample_per_class(raw[raw["label"].isin(CLASSES)], max_per_class)
    d["clean"] = d["text"].map(preprocess)
    d["y"] = le.transform(d["label"])
    return d


t0 = time.perf_counter()
ACTIVE = prep_frame(ACTIVE_RAW, CFG.MAX_PER_CLASS)
print(f"Dataset actif prêt : {len(ACTIVE)} logs ({time.perf_counter() - t0:.1f} s)")
display(ACTIVE[["text", "clean", "label", "y"]].head(4))

# Datasets de la collection (niveaux d'entropie) pour le volet B, préparés à la demande
SYN_NAMES = sorted(k for k in DATASETS if k != REAL_NAME)
PREPS: dict[str, pd.DataFrame] = {}


def get_prep(name: str) -> pd.DataFrame:
    if name not in PREPS:
        raw = ACTIVE_RAW if name == ACTIVE_NAME else clean_frame(DATASETS[name])
        PREPS[name] = prep_frame(raw, CFG.MAX_PER_CLASS_ROBUST)
    return PREPS[name]


print("Datasets disponibles pour le volet B (entropie croissante) :", SYN_NAMES)

## 3. Protocole open-world : construction des 6 plis

### 3.1 Formation des plis
On vérifie que les plis forment une **partition** des 30 classes (chaque classe est inconnue exactement une fois). En mode `family`, les familles de classes (préfixe avant le premier `-`) sont placées ensemble dans un même pli par un remplissage glouton.

In [ ]:
def family_of(c: str) -> str:
    return c.split("-")[0]


def make_folds(classes, n_folds: int, k: int, mode: str, seed: int = CFG.SEED) -> list[list[str]]:
    classes = list(classes)
    if mode == "random":
        perm = np.random.default_rng(seed).permutation(classes)
        folds = [sorted(map(str, f)) for f in np.array_split(perm, n_folds)]
    elif mode == "family":
        fams: dict[str, list[str]] = {}
        for c in sorted(classes):
            fams.setdefault(family_of(c), []).append(c)
        folds = [[] for _ in range(n_folds)]
        for fam in sorted(fams.values(), key=lambda v: -len(v)):
            target = next((f for f in folds if len(f) + len(fam) <= k), None) or min(folds, key=len)
            target.extend(fam)
        folds = [sorted(f) for f in folds]
    else:
        raise ValueError(mode)
    assert sorted(sum(folds, [])) == sorted(classes), "les plis ne forment pas une partition des classes"
    return folds


FOLDS = {m: make_folds(CLASSES, CFG.N_FOLDS, CFG.UNKNOWN_PER_FOLD, m) for m in CFG.FOLD_MODES}
for mode, folds in FOLDS.items():
    print(f"\n=== mode « {mode} » ===")
    for i, f in enumerate(folds, 1):
        print(f"  pli {i} (inconnues) : {', '.join(f)}")

### 3.2 Découpage des classes connues
Pour chaque pli : les logs des 25 classes connues sont séparés en **entraînement / validation / test** (60/15/25 % par défaut, stratifié par classe). Les logs des 5 classes inconnues sont gardés à part (plafonnés par classe), jamais utilisés avant le test.

> **Limite à connaître** : le découpage des classes *connues* est aléatoire au niveau des logs, comme dans l'article. Il mesure donc la précision *closed-set* de façon optimiste (variantes d'un même gabarit des deux côtés, cf. phase 1). Cela n'affecte pas la détection de l'inconnu, qui porte sur des **classes** jamais vues.

In [ ]:
def split_known(df: pd.DataFrame, known_ids, seed: int):
    d = df[df["y"].isin(known_ids)]
    tr, tmp = train_test_split(d, test_size=1 - CFG.SPLIT[0], stratify=d["y"], random_state=seed)
    va, te = train_test_split(tmp, test_size=CFG.SPLIT[2] / (CFG.SPLIT[1] + CFG.SPLIT[2]), stratify=tmp["y"], random_state=seed)
    return tr.reset_index(drop=True), va.reset_index(drop=True), te.reset_index(drop=True)


def unknown_frame(df: pd.DataFrame, unk_ids, seed: int) -> pd.DataFrame:
    return sample_per_class(df[df["y"].isin(unk_ids)], CFG.MAX_EVAL_PER_CLASS, seed)


# Démonstration sur le premier pli du premier mode
unk0 = FOLDS[CFG.FOLD_MODES[0]][0]
unk_ids0 = le.transform(unk0)
known_ids0 = [i for i in range(K) if i not in unk_ids0]
tr0, va0, te0 = split_known(ACTIVE, known_ids0, CFG.SEED)
un0 = unknown_frame(ACTIVE, unk_ids0, CFG.SEED)
print(pd.DataFrame({"jeu": ["train (connues)", "val (connues)", "test (connues)", "test (INCONNUES)"],
                    "logs": [len(tr0), len(va0), len(te0), len(un0)],
                    "classes": [tr0["y"].nunique(), va0["y"].nunique(), te0["y"].nunique(), un0["y"].nunique()]}).to_string(index=False))
assert not (set(tr0["y"]) & set(un0["y"])), "fuite : une classe inconnue figure dans l'entraînement"
print("\nContrôle : aucune classe inconnue dans l'entraînement/validation.")

## 4. Détecteurs d'inconnu et métriques

Un classifieur *closed-set* range toujours un log dans l'une des 25 classes. Pour détecter l'inconnu, on calcule un **score de confiance** par log ; un log dont la confiance est sous un seuil τ est rejeté (« inconnu »).

| Détecteur | Principe | Remarque |
|---|---|---|
| **MSP** (*max softmax probability*) | confiance = $\max_c \mathrm{softmax}(z)_c$ | référence classique ; sensible à la calibration |
| **MaxLogit** | confiance = $\max_c z_c$ | évite la saturation du softmax |
| **Energy** | confiance = $T\log\sum_c e^{z_c/T}$ | agrège tous les logits |
| **Mahalanobis** | confiance = $-\min_c (z-\mu_c)^\top\Sigma^{-1}(z-\mu_c)$ sur une projection **LSA** (SVD 100 dimensions) du TF-IDF | suppose **une gaussienne par classe** : hypothèse fragile si une classe regroupe plusieurs gabarits (cas de SIEVE) |
| **kNN** | confiance = similarité cosinus moyenne aux 5 plus proches logs d'entraînement, dans le même espace LSA | non paramétrique : adapté aux classes multi-modales ; coûteux à grande échelle |

Pour `LinearSVC`, les $z_c$ sont les marges de la fonction de décision (ce ne sont pas des logits calibrés : d'où l'intérêt de la calibration de la solution S2).

**Seuil τ** : 5ᵉ percentile des confiances de la **validation des classes connues** (donc 95 % des connus acceptés), sans jamais regarder les inconnues.

In [ ]:
from scipy.special import logsumexp, softmax
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, roc_auc_score, roc_curve


def conf_from_logits(lg: np.ndarray, T: float = 1.0) -> dict[str, np.ndarray]:
    z = lg / T
    return {"MSP": softmax(z, axis=1).max(1), "MaxLogit": z.max(1), "Energy": T * logsumexp(z, axis=1)}


class MahalanobisDetector:
    """Distance de Mahalanobis aux classes connues, sur une projection LSA du TF-IDF."""

    def __init__(self, n_components: int = 100, seed: int = CFG.SEED):
        self.n_components, self.seed = n_components, seed

    def fit(self, X, y):
        from sklearn.decomposition import TruncatedSVD
        nc = int(max(2, min(self.n_components, X.shape[1] - 1, X.shape[0] - 1)))
        self.svd = TruncatedSVD(n_components=nc, random_state=self.seed).fit(X)
        Z = self.svd.transform(X)
        self.classes_ = np.unique(y)
        self.mu = np.stack([Z[y == c].mean(0) for c in self.classes_])
        Zc = Z - self.mu[np.searchsorted(self.classes_, y)]
        cov = (Zc.T @ Zc) / len(Zc) + 1e-3 * np.eye(Z.shape[1])
        self.P = np.linalg.inv(cov)
        return self

    def score(self, X) -> np.ndarray:
        Z = self.svd.transform(X)
        d = np.stack([(((Z - m) @ self.P) * (Z - m)).sum(1) for m in self.mu], axis=1)
        return -d.min(1)


class KNNDetector:
    """Similarité cosinus moyenne aux k plus proches logs d'entraînement, dans l'espace LSA (non paramétrique)."""

    def __init__(self, svd, X_train, k: int = 5, chunk: int = 1000):
        self.svd, self.k, self.chunk = svd, k, chunk
        Z = svd.transform(X_train)
        self.Z = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9)

    def score(self, X) -> np.ndarray:
        Z = self.svd.transform(X)
        Z = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9)
        out = np.empty(len(Z))
        for i in range(0, len(Z), self.chunk):
            S = Z[i:i + self.chunk] @ self.Z.T
            k = min(self.k, S.shape[1])
            out[i:i + self.chunk] = np.partition(S, -k, axis=1)[:, -k:].mean(1)
        return out


def ece_score(conf: np.ndarray, correct: np.ndarray, bins: int = 15) -> float:
    edges = np.linspace(0, 1, bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)


def open_metrics(c_va, c_te, c_un, pred_te, y_te, pred_un) -> dict:
    """Détection de l'inconnu : positif = inconnu ; confiance haute = connu."""
    y_open = np.r_[np.zeros(len(c_te)), np.ones(len(c_un))]
    conf_all = np.r_[c_te, c_un]
    auroc = roc_auc_score(y_open, -conf_all)
    aupr = average_precision_score(y_open, -conf_all)
    fpr, tpr, _ = roc_curve(1 - y_open, conf_all)            # positif = connu
    fpr95 = float(fpr[min(np.searchsorted(tpr, 0.95, side="left"), len(fpr) - 1)])
    tau = np.percentile(c_va, 5)                              # fixé sur la validation des CONNUES uniquement
    kept = float((c_te >= tau).mean())
    rejected = float((c_un < tau).mean())
    yt = np.r_[y_te, -np.ones(len(c_un), dtype=int)]
    yp = np.r_[np.where(c_te >= tau, pred_te, -1), np.where(c_un >= tau, pred_un, -1)]
    return {"auroc": auroc, "aupr": aupr, "fpr95": fpr95, "known_kept": kept, "unknown_rejected": rejected,
            "open_f1": f1_score(yt, yp, average="macro", zero_division=0)}


def eval_open(lg_va, lg_te, lg_un, classes, y_te, detectors, T=1.0, extra=None):
    """Évalue tous les détecteurs à partir des logits ; extra = {nom: (conf_val, conf_test, conf_unk)}."""
    cv, ct, cu = conf_from_logits(lg_va, T), conf_from_logits(lg_te, T), conf_from_logits(lg_un, T)
    for name, (a, b, c) in (extra or {}).items():
        cv[name], ct[name], cu[name] = a, b, c
    pred_te, pred_un = classes[lg_te.argmax(1)], classes[lg_un.argmax(1)]
    closed = {"closed_acc": accuracy_score(y_te, pred_te), "closed_f1": f1_score(y_te, pred_te, average="macro", zero_division=0)}
    rows = []
    for d in detectors:
        m = open_metrics(cv[d], ct[d], cu[d], pred_te, y_te, pred_un)
        m["ece"] = ece_score(ct[d], (pred_te == y_te).astype(float)) if d == "MSP" else np.nan
        rows.append({"detector": d, **closed, **m})
    return rows, ct, cu, pred_un

## 5. Expérience A — open-world sur 6 plis (modèles linéaires)

Pour chaque mode de pli, chaque pli, chaque modèle (`LogisticRegression`, `LinearSVC`) et chaque détecteur : entraînement sur les 25 classes connues (TF-IDF ajusté sur le train seul), calcul des scores sur validation / test connu / inconnues, métriques.

- **LogisticRegression** : logits = `decision_function` ; modèle probabiliste de référence.
- **LinearSVC** : marges ; très rapide.

La cellule affiche l'avancement pli par pli. Temps indicatif sur CPU de portable : quelques minutes par mode avec les réglages par défaut.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC


def make_word_tfidf() -> TfidfVectorizer:
    return TfidfVectorizer(lowercase=False, token_pattern=r"\S+", norm="l2")


def make_clf(name: str):
    if name == "LogisticRegression":
        return LogisticRegression(max_iter=300, random_state=CFG.SEED)
    return LinearSVC(max_iter=5000, random_state=CFG.SEED)


def run_open_experiment(models, modes, detectors) -> tuple[pd.DataFrame, pd.DataFrame]:
    require_eda()
    rows, class_rows = [], []
    for mode in modes:
        for fi, unk in enumerate(FOLDS[mode], 1):
            seed = CFG.SEED + fi
            unk_ids = le.transform(unk)
            known_ids = [i for i in range(K) if i not in unk_ids]
            tr, va, te = split_known(ACTIVE, known_ids, seed)
            un = unknown_frame(ACTIVE, unk_ids, seed)
            vec = make_word_tfidf()
            Xtr = vec.fit_transform(tr["clean"])
            Xva, Xte, Xun = vec.transform(va["clean"]), vec.transform(te["clean"]), vec.transform(un["clean"])
            extra = {}
            if "Mahalanobis" in detectors or "kNN" in detectors:
                maha = MahalanobisDetector().fit(Xtr, tr["y"].to_numpy())
                if "Mahalanobis" in detectors:
                    extra["Mahalanobis"] = (maha.score(Xva), maha.score(Xte), maha.score(Xun))
                if "kNN" in detectors:
                    knn = KNNDetector(maha.svd, Xtr)
                    extra["kNN"] = (knn.score(Xva), knn.score(Xte), knn.score(Xun))
            for mname in models:
                t0 = time.perf_counter()
                clf = make_clf(mname).fit(Xtr, tr["y"])
                lg = {k: clf.decision_function(X) for k, X in (("va", Xva), ("te", Xte), ("un", Xun))}
                res, ct, cu, pred_un = eval_open(lg["va"], lg["te"], lg["un"], clf.classes_, te["y"].to_numpy(), detectors, extra=extra)
                for r in res:
                    rows.append({"mode": mode, "fold": fi, "model": mname, "unknown": ", ".join(unk), **r})
                # AUROC par classe inconnue (détecteur MSP) et classe connue la plus souvent choisie
                y_un = un["y"].to_numpy()
                for u in unk_ids:
                    m = y_un == u
                    auc = roc_auc_score(np.r_[np.zeros(len(ct["MSP"])), np.ones(m.sum())], -np.r_[ct["MSP"], cu["MSP"][m]])
                    top = pd.Series(pred_un[m]).value_counts()
                    class_rows.append({"mode": mode, "fold": fi, "model": mname, "unknown_class": CLASSES[u],
                                       "auroc_MSP": auc, "assigned_to": CLASSES[top.index[0]], "share": top.iloc[0] / m.sum()})
                best = max(res, key=lambda r: r["auroc"])
                print(f"[{mode:<6}|pli {fi}|{mname:<18}] closed acc={res[0]['closed_acc']:.3f} | "
                      f"AUROC MSP={next(r['auroc'] for r in res if r['detector'] == 'MSP'):.3f} | meilleur : {best['detector']} ({best['auroc']:.3f}) "
                      f"| {time.perf_counter() - t0:.0f}s")
    return pd.DataFrame(rows), pd.DataFrame(class_rows)


set_seed()
RES_A, CLS_A = run_open_experiment(CFG.MODELS_A, CFG.FOLD_MODES, CFG.DETECTORS_A)
RES_A.to_csv(OUT / "ow_results_A_folds.csv", index=False)
CLS_A.to_csv(OUT / "ow_results_A_per_class.csv", index=False)

### 5.1 Résultats agrégés (moyenne ± écart-type sur les plis)
`closed_acc`/`closed_f1` : performance sur les **25 classes connues**. `auroc`, `aupr` : détection de l'inconnu (plus haut = mieux). `fpr95` : inconnus acceptés à tort quand 95 % des connus sont acceptés (plus bas = mieux). `known_kept` / `unknown_rejected` : à seuil τ fixé sur la validation. `open_f1` : F1 macro à 26 sorties.

In [ ]:
def summarize(df: pd.DataFrame, by: list[str]) -> pd.DataFrame:
    cols = ["closed_acc", "closed_f1", "auroc", "aupr", "fpr95", "known_kept", "unknown_rejected", "open_f1"]
    g = df.groupby(by)[cols]
    out = g.mean().round(4).astype(str) + " ± " + g.std().fillna(0).round(4).astype(str)
    return out


SUM_A = summarize(RES_A, ["mode", "model", "detector"])
display(SUM_A)
SUM_A.to_csv(OUT / "ow_summary_A.csv")

g = sns.catplot(data=RES_A, x="detector", y="auroc", hue="model", col="mode", kind="bar", errorbar="sd",
                height=4, aspect=1.3, order=CFG.DETECTORS_A)
g.set_axis_labels("Détecteur", "AUROC (inconnu vs connu)")
g.figure.suptitle("Détection de l'inconnu — moyenne ± écart-type sur les plis", y=1.03)
for ax in g.axes.ravel():
    ax.axhline(0.5, color="grey", ls="--", lw=0.8)
g.figure.savefig(OUT / "figures" / "ow_A_auroc_by_detector.png", dpi=150, bbox_inches="tight")
plt.show()

g = sns.catplot(data=RES_A, x="detector", y="fpr95", hue="model", col="mode", kind="bar", errorbar="sd",
                height=4, aspect=1.3, order=CFG.DETECTORS_A)
g.set_axis_labels("Détecteur", "FPR@95 %TPR (plus bas = mieux)")
g.figure.savefig(OUT / "ow_A_fpr95.png".replace("ow_A", "figures/ow_A"), dpi=150, bbox_inches="tight")
plt.show()

### 5.2 Quelles classes inconnues sont les plus difficiles ?
AUROC *par classe inconnue* (détecteur MSP) et classe connue vers laquelle ses logs sont le plus souvent aiguillés. C'est l'analyse la plus utile pour la discussion : elle montre si les échecs viennent de **ressemblances sémantiques** (`file-deleted` rangé dans `directory-deleted`…) plutôt que d'un défaut du détecteur.

In [ ]:
cls_tab = (CLS_A.groupby(["mode", "model", "unknown_class"])
                .agg(auroc_MSP=("auroc_MSP", "mean"), assigned_to=("assigned_to", "first"), share=("share", "mean"))
                .round(3).reset_index())
for mode in CFG.FOLD_MODES:
    sub = cls_tab[(cls_tab["mode"] == mode) & (cls_tab["model"] == CFG.MODELS_A[0])].sort_values("auroc_MSP")
    print(f"\n=== mode « {mode} » — {CFG.MODELS_A[0]} — classes les plus difficiles à reconnaître comme inconnues ===")
    display(sub.head(8).drop(columns=["mode", "model"]))

pv = cls_tab[cls_tab["mode"] == CFG.FOLD_MODES[0]].pivot(index="unknown_class", columns="model", values="auroc_MSP").sort_values(CFG.MODELS_A[0])
fig, ax = plt.subplots(figsize=(6, 0.3 * len(pv) + 1.5))
sns.heatmap(pv, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0.3, vmax=1, ax=ax)
ax.set(title=f"AUROC (MSP) par classe inconnue — mode « {CFG.FOLD_MODES[0]} »", ylabel="", xlabel="")
plt.tight_layout()
fig.savefig(OUT / "figures" / "ow_A_per_class_auroc.png", dpi=150)
plt.show()
cls_tab.to_csv(OUT / "ow_per_class_table.csv", index=False)

**Analyse (à rédiger)** :
1. Quel détecteur est le meilleur ? Le softmax (MSP) est-il battu par l'énergie ou le kNN ? Mahalanobis donne-t-il un AUROC proche de 0,5 (voire inférieur) ? Si oui, vérifier l'hypothèse « classes multi-modales » : une classe SIEVE regroupe plusieurs gabarits, donc sa moyenne se situe *entre* des amas et un log connu peut être plus éloigné de cette moyenne qu'un log inconnu.
2. Écart entre les modes `random` et `family` : si le mode `family` est nettement pire, l'échec vient surtout de **classes inconnues proches de classes connues** (ambiguïté sémantique), pas d'un mauvais détecteur.
3. LR (probabiliste) contre LinearSVC (marges non calibrées) : la calibration compte-t-elle pour MSP ?
4. Quelles classes sont rarement détectées comme inconnues, et vers quelles classes connues sont-elles aiguillées ? Comparer avec les confusions *closed-set* de l'article (`directory-deleted`/`file-deleted`, etc.).
5. À seuil τ fixé sur la validation : quelle part d'inconnus est rejetée, et quelle part de connus est conservée ?

## 6. Volet B — deux solutions de robustesse

### 6.1 Principe et protocole commun
Les trois stratégies à comparer à la **référence** (TF-IDF mots, entraînement sur `SIEVE_00` seul, sans calibration) :

| Stratégie | Données d'entraînement | Features | Calibration |
|---|---|---|---|
| **Baseline** | ACTIVE seul | mots | non |
| **S1** | mélange multi-entropie + 50 % de logs bruités | mots | non |
| **S2** | ACTIVE seul | mots + caractères (3-4 grammes) | température |
| **S1+S2** | mélange multi-entropie + bruit | mots + caractères | température |

**Budget constant** : toutes les stratégies s'entraînent sur **le même nombre de logs** (celui de la baseline). Le mélange S1 prélève une part égale dans chaque niveau d'entropie ; le gain éventuel vient donc de la *diversité*, pas du *volume*.

**Bruit d'entraînement (S1)** : pour 50 % des logs, un niveau $p\in\{0{,}1;0{,}2;0{,}3\}$ et un type de perturbation (échange *ou* suppression de tokens) tirés au hasard. La suppression aléatoire (RD) est volontairement incluse bien que les auteurs de SPICE l'aient exclue pour la génération : ici elle sert à **entraîner la robustesse**, pas à fabriquer des logs de référence.

**Calibration (S2)** : la température $T$ est ajustée sur la validation des classes connues (minimisation de l'entropie croisée). Elle ne change pas la classe prédite, seulement les confiances.

### 6.2 Conditions d'évaluation
- **dans la distribution** : test de ACTIVE ;
- **entropie élevée** : logs du dataset d'entropie la plus forte, **jamais vu** à l'entraînement (S1 n'utilise que les niveaux bas) ;
- **bruit** : test de ACTIVE perturbé (échange + suppression, $p=0{,}2$).

> **Ce que ce volet ne mesure pas** : tous les niveaux d'entropie de SIEVE viennent des **mêmes 440 gabarits**. Les conditions « entropie élevée » et « bruit » testent la robustesse aux *perturbations de surface*, pas la généralisation à de nouveaux gabarits : c'est le rôle du volet A (classes entières inconnues) et du test sur logs réels (phase 1).

In [ ]:
def perturb_clean(s: str, p_swap: float, p_del: float, rng: np.random.Generator) -> str:
    toks = s.split()
    if len(toks) >= 2 and p_swap > 0:
        for _ in range(max(1, int(round(p_swap * len(toks))))):
            i, j = rng.integers(0, len(toks), 2)
            toks[i], toks[j] = toks[j], toks[i]
    if p_del > 0 and len(toks) > 1:
        toks = [t for t in toks if rng.random() > p_del] or toks[:1]
    return " ".join(toks)


def perturb_frame(df: pd.DataFrame, p_swap: float, p_del: float, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    d = df.copy()
    d["clean"] = [perturb_clean(s, p_swap, p_del, rng) for s in d["clean"]]
    return d


# Niveaux d'entropie utilisés pour S1 (bas) et pour le test décalé (le plus haut, jamais vu)
S1_SETS = ([ACTIVE_NAME] + [n for n in SYN_NAMES if n != ACTIVE_NAME])[:3]
SHIFT_NAME = next((n for n in reversed(SYN_NAMES) if n not in S1_SETS), None)
print("S1 (multi-entropie) s'entraîne sur :", S1_SETS)
print("Test « entropie élevée » sur :", SHIFT_NAME, "(jamais vu à l'entraînement)")
if len(S1_SETS) < 2:
    print("[info] un seul niveau d'entropie disponible : S1 se réduit à l'augmentation par bruit.")
if SHIFT_NAME is None:
    print("[info] pas de dataset décalé disponible : la condition « entropie élevée » sera ignorée.")

STRATS = {
    "Baseline": dict(multi=False, noise=False, char=False, calib=False),
    "S1": dict(multi=True, noise=True, char=False, calib=False),
    "S2": dict(multi=False, noise=False, char=True, calib=True),
    "S1+S2": dict(multi=True, noise=True, char=True, calib=True),
}


def build_train(spec: dict, tr_base: pd.DataFrame, known_ids, seed: int) -> pd.DataFrame:
    """Construit le jeu d'entraînement d'une stratégie, TOUJOURS à taille égale à tr_base (budget constant)."""
    rng = np.random.default_rng(seed)
    d = tr_base
    if spec["multi"] and len(S1_SETS) > 1:
        parts = [tr_base] + [get_prep(n)[get_prep(n)["y"].isin(known_ids)] for n in S1_SETS if n != ACTIVE_NAME]
        per = len(tr_base) // len(parts)
        d = pd.concat([p.sample(n=min(per, len(p)), random_state=seed) for p in parts], ignore_index=True)
    if spec["noise"]:
        d = d.copy()
        clean = d["clean"].tolist()
        for i in np.where(rng.random(len(d)) < 0.5)[0]:
            p = float(rng.choice([0.1, 0.2, 0.3]))
            swap = rng.random() < 0.5                      # moitié échange, moitié suppression
            clean[i] = perturb_clean(clean[i], p if swap else 0.0, 0.0 if swap else p, rng)
        d["clean"] = clean
    return d


def make_features(char: bool):
    from sklearn.pipeline import FeatureUnion
    word = make_word_tfidf()
    if not char:
        return word
    chars = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4), min_df=2, max_features=150000,
                            sublinear_tf=True, lowercase=False)
    return FeatureUnion([("w", word), ("c", chars)])


def fit_temperature(logits: np.ndarray, y: np.ndarray, classes: np.ndarray) -> float:
    idx = np.searchsorted(classes, y)
    best, best_T = np.inf, 1.0
    for T in np.exp(np.linspace(np.log(0.05), np.log(20), 60)):
        z = logits / T
        nll = -(z[np.arange(len(z)), idx] - logsumexp(z, axis=1)).mean()
        if nll < best:
            best, best_T = nll, float(T)
    return best_T


def fit_strategy(spec: dict, tr: pd.DataFrame, va: pd.DataFrame, model_name: str) -> dict:
    feats = make_features(spec["char"])
    clf = make_clf(model_name).fit(feats.fit_transform(tr["clean"]), tr["y"])
    T = 1.0
    if spec["calib"]:
        T = fit_temperature(clf.decision_function(feats.transform(va["clean"])), va["y"].to_numpy(), clf.classes_)
    return {"feats": feats, "clf": clf, "T": T}


def strat_logits(model: dict, df: pd.DataFrame) -> np.ndarray:
    return model["clf"].decision_function(model["feats"].transform(df["clean"]))

### 6.3 Test rapide de la stratégie de bruit
Vérification que la perturbation fait ce qu'elle doit (et que la taille d'entraînement est bien constante).

In [ ]:
base = get_prep(ACTIVE_NAME)
tr_b, va_b, te_b = split_known(base, list(range(K)), CFG.SEED)
ex = tr_b["clean"].iloc[0]
rng_demo = np.random.default_rng(1)
print("original :", ex)
print("échange  :", perturb_clean(ex, 0.3, 0.0, rng_demo))
print("suppress.:", perturb_clean(ex, 0.0, 0.3, rng_demo))
for name, spec in STRATS.items():
    d = build_train(spec, tr_b, list(range(K)), CFG.SEED)
    print(f"{name:<9} taille d'entraînement = {len(d)} (référence {len(tr_b)}) | classes = {d['y'].nunique()}")

### 6.4 Volet B1 — robustesse *closed-set* (30 classes) : balayage de l'entropie et du bruit
On entraîne chaque stratégie sur toutes les classes (budget égal), puis on mesure le **F1 macro** :
- sur chaque niveau d'entropie de la collection (balayage $p$ croissant) ;
- sur le test de ACTIVE perturbé avec une intensité croissante (échange ou suppression de tokens).

Les niveaux d'entropie vus par S1 à l'entraînement sont signalés : seule la zone « non vue » compare équitablement les stratégies.

In [ ]:
def sample_per_class(df, n, seed=CFG.SEED):
    rng = np.random.default_rng(seed)

    if df.empty:
        raise ValueError("Dataset vide après filtrage des classes.")

    idx = []

    for _, ix in df.groupby("label").indices.items():
        ix = np.asarray(ix)
        idx.append(
            rng.choice(ix, size=min(n, len(ix)), replace=False)
        )

    if not idx:
        raise ValueError("Aucune donnée disponible pour l'échantillonnage.")

    return df.iloc[np.sort(np.concatenate(idx))].reset_index(drop=True)


def prep_frame(raw: pd.DataFrame, max_per_class: int | None) -> pd.DataFrame:
    d = raw.copy()

    if not {"label", "text"}.issubset(d.columns):
        raise ValueError("Colonnes 'label' et 'text' requises.")

    d["label"] = d["label"].astype(str).str.strip().str.lower()
    classes_norm = [str(c).strip().lower() for c in CLASSES]

    d = d[d["label"].isin(classes_norm)].copy()

    if d.empty:
        raise ValueError(
            f"Aucune classe reconnue. Labels disponibles : "
            f"{raw['label'].dropna().unique().tolist()}"
        )

    d = sample_per_class(d, max_per_class)
    d["clean"] = d["text"].map(preprocess)
    d["y"] = le.transform(d["label"])

    return d


def get_prep(name: str) -> pd.DataFrame:
    if name not in PREPS:
        raw = ACTIVE_RAW if name == ACTIVE_NAME else clean_frame(DATASETS[name])

        if not {"label", "text"}.issubset(raw.columns):
            raise ValueError(f"Dataset '{name}' : colonnes label/text absentes.")

        raw = raw.copy()
        raw["label"] = raw["label"].astype(str).str.strip().str.lower()

        classes_norm = [str(c).strip().lower() for c in CLASSES]
        raw = raw[raw["label"].isin(classes_norm)].copy()

        if raw.empty:
            raise ValueError(
                f"Dataset '{name}' invalide : aucune classe attendue. "
                f"Labels trouvés : "
                f"{DATASETS[name]['label'].dropna().unique().tolist()}"
            )

        PREPS[name] = prep_frame(raw, CFG.MAX_PER_CLASS_ROBUST)

    return PREPS[name]


def run_b1() -> tuple[pd.DataFrame, pd.DataFrame]:
    require_eda()

    all_ids = list(range(K))
    sweep, noise = [], []

    for sname, spec in STRATS.items():
        t0 = time.perf_counter()

        tr_s = build_train(spec, tr_b, all_ids, CFG.SEED)
        model = fit_strategy(spec, tr_s, va_b, CFG.B_MODEL)

        for name in SYN_NAMES:
            ev = te_b if name == ACTIVE_NAME else get_prep(name)

            pred = model["clf"].classes_[
                strat_logits(model, ev).argmax(1)
            ]

            sweep.append({
                "strategy": sname,
                "dataset": name,
                "seen_by_S1": name in S1_SETS,
                "f1_macro": f1_score(
                    ev["y"], pred,
                    average="macro",
                    zero_division=0
                )
            })

        for kind in ("swap", "delete"):
            for p in (0.0, 0.1, 0.2, 0.3, 0.5):
                ev = perturb_frame(
                    te_b,
                    p if kind == "swap" else 0.0,
                    p if kind == "delete" else 0.0,
                    CFG.SEED
                )

                pred = model["clf"].classes_[
                    strat_logits(model, ev).argmax(1)
                ]

                noise.append({
                    "strategy": sname,
                    "perturbation": kind,
                    "p": p,
                    "f1_macro": f1_score(
                        ev["y"], pred,
                        average="macro",
                        zero_division=0
                    )
                })

        print(
            f"[B1|{sname:<9}] T={model['T']:.2f} | "
            f"{time.perf_counter() - t0:.0f}s"
        )

    return pd.DataFrame(sweep), pd.DataFrame(noise)


# Préparation des datasets valides
PREPS.clear()

SYN_NAMES = [
    name for name in SYN_NAMES
    if name == ACTIVE_NAME
    or (
        name in DATASETS
        and {"label", "text"}.issubset(DATASETS[name].columns)
        and DATASETS[name]["label"].astype(str).str.strip().str.lower()
        .isin([str(c).strip().lower() for c in CLASSES]).any()
    )
]

if ACTIVE_NAME not in SYN_NAMES:
    SYN_NAMES.append(ACTIVE_NAME)

# Exécution
B1_SWEEP, B1_NOISE = run_b1()

# Sauvegarde
B1_SWEEP.to_csv(OUT / "ow_B1_entropy_sweep.csv", index=False)
B1_NOISE.to_csv(OUT / "ow_B1_noise_sweep.csv", index=False)

# Tableau comparatif
display(
    B1_SWEEP.pivot(
        index="strategy",
        columns="dataset",
        values="f1_macro"
    ).round(4)
)

# Graphiques
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))

sns.lineplot(
    data=B1_SWEEP,
    x="dataset",
    y="f1_macro",
    hue="strategy",
    marker="o",
    ax=axes[0]
)

axes[0].set(
    title="F1 macro selon le niveau d'entropie",
    xlabel="",
    ylabel="F1 macro"
)
axes[0].tick_params(axis="x", rotation=30)

for ax, kind, title in zip(
    axes[1:],
    ("swap", "delete"),
    ("Échange de tokens (RS)", "Suppression de tokens (RD)")
):
    sns.lineplot(
        data=B1_NOISE[B1_NOISE["perturbation"] == kind],
        x="p",
        y="f1_macro",
        hue="strategy",
        marker="o",
        ax=ax
    )

    ax.set(title=title, xlabel="Intensité p", ylabel="F1 macro")

plt.tight_layout()

fig.savefig(
    OUT / "figures" / "ow_B1_robustness.png",
    dpi=150
)

plt.show()

**Analyse (à rédiger)** :
- Sur les niveaux d'entropie **non vus** (ou sous bruit fort), S1 et/ou S2 dégradent-ils moins que la baseline ? Chiffrer l'écart à $p$ élevé.
- La suppression de tokens est-elle plus destructrice que l'échange (hypothèse des auteurs de SPICE) ? Les n-grammes de caractères (S2) la compensent-ils ?
- Un gain de S1 à budget constant est un argument en faveur de la **diversité** d'entropie plutôt que du volume.

### 6.5 Volet B2 — les solutions améliorent-elles la détection de l'inconnu ?
On rejoue le protocole open-world (mode `random`, 6 plis) pour **chaque stratégie**, dans les **trois conditions** (dans la distribution, entropie élevée, bruit). Mesures : F1 macro closed-set sur les 25 classes connues, AUROC et FPR@95 %TPR de la détection de l'inconnu (MSP et Energy), ECE (qualité de la calibration du MSP).

In [ ]:
def run_b2(folds_idx=None, mode: str = "random") -> pd.DataFrame:
    require_eda()

    rows = []
    folds = FOLDS[mode]

    # Nettoyage du cache
    PREPS.clear()

    # Préparation du dataset principal
    base = get_prep(ACTIVE_NAME)

    # Chargement sécurisé du dataset de shift
    shift = None

    if SHIFT_NAME:
        try:
            shift = get_prep(SHIFT_NAME)
        except (ValueError, KeyError, FileNotFoundError) as e:
            print(f"[WARNING] Dataset de shift ignoré : {e}")
            shift = None

    for fi, unk in enumerate(folds, 1):

        if folds_idx and fi not in folds_idx:
            continue

        seed = CFG.SEED + 100 + fi

        unk_ids = le.transform(unk)
        known_ids = [i for i in range(K) if i not in unk_ids]

        tr, va, te = split_known(base, known_ids, seed)
        un = unknown_frame(base, unk_ids, seed)

        conds = {
            "dans la distribution": (te, un)
        }

        if shift is not None:
            shift_known = shift[shift["y"].isin(known_ids)]

            if not shift_known.empty:
                shift_eval = sample_per_class(
                    shift_known,
                    CFG.MAX_EVAL_PER_CLASS,
                    seed
                )

                shift_unknown = unknown_frame(
                    shift,
                    unk_ids,
                    seed
                )

                conds["entropie élevée"] = (
                    shift_eval,
                    shift_unknown
                )

        conds["bruit"] = (
            perturb_frame(te, 0.2, 0.2, seed),
            perturb_frame(un, 0.2, 0.2, seed + 1)
        )

        for sname, spec in STRATS.items():

            t0 = time.perf_counter()

            model = fit_strategy(
                spec,
                build_train(spec, tr, known_ids, seed),
                va,
                CFG.B_MODEL
            )

            classes = model["clf"].classes_
            T = model["T"]

            lg_va = strat_logits(model, va)

            for cname, (kn, uk) in conds.items():

                res, *_ = eval_open(
                    lg_va,
                    strat_logits(model, kn),
                    strat_logits(model, uk),
                    classes,
                    kn["y"].to_numpy(),
                    CFG.DETECTORS_B,
                    T=T
                )

                for r in res:
                    rows.append({
                        "fold": fi,
                        "strategy": sname,
                        "condition": cname,
                        "T": T,
                        **r
                    })

            print(
                f"[B2|pli {fi}|{sname:<9}] "
                f"T={T:.2f} | "
                f"{time.perf_counter() - t0:.0f}s"
            )

    return pd.DataFrame(rows)


# Exécution
set_seed()

RES_B2 = run_b2(CFG.B_FOLDS)

# Sauvegarde
RES_B2.to_csv(
    OUT / "ow_results_B2.csv",
    index=False
)

display(RES_B2)

In [ ]:
SUM_B2 = summarize(RES_B2, ["condition", "detector", "strategy"])
display(SUM_B2)
SUM_B2.to_csv(OUT / "ow_summary_B2.csv")

order = list(STRATS)
g = sns.catplot(data=RES_B2[RES_B2["detector"] == "MSP"], x="strategy", y="auroc", hue="strategy", col="condition",
                kind="bar", errorbar="sd", order=order, height=4, aspect=1.0, legend=False)
g.set_axis_labels("", "AUROC (MSP)")
g.figure.suptitle("Détection de l'inconnu selon la stratégie et la condition", y=1.03)
g.figure.savefig(OUT / "figures" / "ow_B2_auroc.png", dpi=150, bbox_inches="tight")
plt.show()

g = sns.catplot(data=RES_B2[RES_B2["detector"] == "MSP"], x="strategy", y="closed_f1", hue="strategy", col="condition",
                kind="bar", errorbar="sd", order=order, height=4, aspect=1.0, legend=False)
g.set_axis_labels("", "F1 macro (25 classes connues)")
g.figure.savefig(OUT / "figures" / "ow_B2_closed_f1.png", dpi=150, bbox_inches="tight")
plt.show()

cal = RES_B2[(RES_B2["detector"] == "MSP") & (RES_B2["condition"] == "dans la distribution")].groupby("strategy")[["ece", "T"]].mean().round(4)
print("Qualité de calibration du MSP (ECE, plus bas = mieux) et température apprise :")
display(cal.reindex(order))

**Analyse (à rédiger)** :
1. La calibration (S2) améliore-t-elle l'AUROC/FPR@95 du **MSP** ? Elle ne change pas la classe prédite : tout gain vient d'un meilleur *classement* des confiances. Comparer l'ECE avant/après.
2. S1 (multi-entropie + bruit) réduit-elle la chute de l'AUROC dans les conditions « entropie élevée » et « bruit » ?
3. Y a-t-il un compromis ? Une solution qui améliore la robustesse mais dégrade l'accuracy *closed-set* (ou l'inverse) doit être signalée.
4. Comparer l'ampleur de ces effets à l'écart-type entre plis : un gain inférieur à l'écart-type n'est pas démontré.

## 7. Volet C (optionnel, Deep Learning) — LSTM en monde ouvert
Même protocole (mode `random`, 6 plis), mais le classifieur est un **LSTM** (plongement 50, 2 couches de 128 unités, comme en phase 1) entraîné sur les 25 classes connues. On calcule les mêmes détecteurs (MSP, MaxLogit, Energy) et on compare aux modèles linéaires du volet A. Le LSTM est un modèle *profond* : sa confiance est souvent excessive (sur-confiance), ce qui rend le MSP trompeur — c'est précisément le phénomène que ce volet permet d'observer.

*Durée* : plusieurs minutes par pli sur GPU, nettement plus sur CPU. Réduire `CFG.DEEP_MAX_TRAIN`, `CFG.DEEP_EPOCHS` ou `CFG.DEEP_FOLDS` (ex. `[1, 2]`) pour un essai.

In [ ]:
if HAS_TORCH:
    import torch.nn as nn

    class LSTMClassifier(nn.Module):
        def __init__(self, vocab_size, n_classes, emb=50, hidden=128, layers=2, dropout=0.3):
            super().__init__()
            self.emb = nn.Embedding(vocab_size, emb, padding_idx=0)
            self.rnn = nn.LSTM(emb, hidden, layers, batch_first=True, dropout=dropout)
            self.drop = nn.Dropout(dropout)
            self.fc = nn.Linear(hidden, n_classes)

        def forward(self, x, lengths):
            packed = nn.utils.rnn.pack_padded_sequence(self.drop(self.emb(x)), lengths.cpu(), batch_first=True, enforce_sorted=False)
            _, (h, _) = self.rnn(packed)
            return self.fc(self.drop(h[-1]))

    def lstm_open_fold(tr: pd.DataFrame, eval_frames: dict, known_ids, seed: int):
        """Entraîne un LSTM sur les classes connues ; renvoie les logits de chaque jeu d'évaluation."""
        set_seed(seed)
        classes = np.array(sorted(known_ids))
        cidx = {int(c): i for i, c in enumerate(classes)}
        tr = sample_per_class(tr, max(1, CFG.DEEP_MAX_TRAIN // len(classes)), seed)
        counts = Counter(w for t in tr["clean"] for w in t.split())
        vocab = {"<pad>": 0, "<unk>": 1}
        for w, c in counts.most_common():
            if c >= 2:
                vocab[w] = len(vocab)
        max_len = int(np.clip(np.percentile(tr["clean"].str.split().str.len(), 99), 8, 64))

        def encode(s):
            return [vocab.get(w, 1) for w in s.split()][:max_len] or [1]

        def batches(df, labels=None, shuffle=False, bs=256):
            ids = [encode(s) for s in df["clean"]]
            order = np.random.permutation(len(ids)) if shuffle else np.arange(len(ids))
            for i in range(0, len(ids), bs):
                sel = order[i:i + bs]
                seqs = [ids[j] for j in sel]
                lengths = torch.tensor([len(s) for s in seqs])
                x = torch.zeros(len(seqs), int(lengths.max()), dtype=torch.long)
                for r, s in enumerate(seqs):
                    x[r, :len(s)] = torch.tensor(s)
                y = torch.tensor([cidx[int(v)] for v in labels[sel]]) if labels is not None else None
                yield x, lengths, y

        model = LSTMClassifier(len(vocab), len(classes)).to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=2e-3)
        y_tr = tr["y"].to_numpy()
        for ep in range(CFG.DEEP_EPOCHS):
            model.train()
            for x, L, y in batches(tr, y_tr, shuffle=True):
                opt.zero_grad()
                loss = nn.functional.cross_entropy(model(x.to(DEVICE), L), y.to(DEVICE))
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
        model.eval()
        out = {}
        with torch.inference_mode():
            for name, df in eval_frames.items():
                out[name] = np.concatenate([model(x.to(DEVICE), L).cpu().numpy() for x, L, _ in batches(df)])
        return out, classes

    def run_deep_open() -> pd.DataFrame:
        require_eda()
        rows = []
        for fi, unk in enumerate(FOLDS["random"], 1):
            if CFG.DEEP_FOLDS and fi not in CFG.DEEP_FOLDS:
                continue
            seed = CFG.SEED + fi
            unk_ids = le.transform(unk)
            known_ids = [i for i in range(K) if i not in unk_ids]
            tr, va, te = split_known(ACTIVE, known_ids, seed)
            un = unknown_frame(ACTIVE, unk_ids, seed)
            t0 = time.perf_counter()
            lg, classes = lstm_open_fold(tr, {"va": va, "te": te, "un": un}, known_ids, seed)
            res, *_ = eval_open(lg["va"], lg["te"], lg["un"], classes, te["y"].to_numpy(), ["MSP", "MaxLogit", "Energy"])
            for r in res:
                rows.append({"mode": "random", "fold": fi, "model": "LSTM", "unknown": ", ".join(unk), **r})
            print(f"[LSTM|pli {fi}] closed acc={res[0]['closed_acc']:.3f} | AUROC MSP={res[0]['auroc']:.3f} | {time.perf_counter() - t0:.0f}s")
        return pd.DataFrame(rows)

In [ ]:
RES_C = pd.DataFrame()
if CFG.RUN_DEEP and HAS_TORCH:
    RES_C = run_deep_open()
    RES_C.to_csv(OUT / "ow_results_C_lstm.csv", index=False)
    comp = pd.concat([RES_A[RES_A["mode"] == "random"], RES_C], ignore_index=True)
    comp = comp[comp["detector"].isin(["MSP", "MaxLogit", "Energy"])]
    display(summarize(comp, ["model", "detector"]))
    g = sns.catplot(data=comp, x="detector", y="auroc", hue="model", kind="bar", errorbar="sd", height=4, aspect=1.5)
    g.set_axis_labels("Détecteur", "AUROC (inconnu vs connu)")
    g.figure.savefig(OUT / "figures" / "ow_C_lstm_vs_linear.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("[skip] Volet C ignoré (PyTorch indisponible ou CFG.RUN_DEEP=False).")

**Analyse (à rédiger)** : le LSTM est-il plus ou moins bon que LR/LinearSVC pour détecter l'inconnu, malgré une accuracy *closed-set* comparable ? Un MSP élevé sur des logs inconnus (sur-confiance) est le signe classique des réseaux profonds : l'énergie ou la distance dans l'espace des features corrige-t-elle cela ?

## 8. Export et conclusion
Tous les tableaux et figures sont dans `./outputs/` (préfixe `ow_`).

In [ ]:
cfg_dump = {k: (v if isinstance(v, (int, float, str, bool, type(None), list, tuple)) else str(v)) for k, v in vars(CFG).items()}
(OUT / "config_phase2.json").write_text(json.dumps({"CFG": cfg_dump, "folds": FOLDS, "S1_SETS": S1_SETS, "SHIFT": SHIFT_NAME,
                                                    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}, indent=2, ensure_ascii=False), encoding="utf-8")
for p in sorted(OUT.glob("ow_*")):
    print("  ", p.name)
for p in sorted((OUT / "figures").glob("ow_*")):
    print("  figures/" + p.name)

### 8.1 Trame de conclusion (à compléter avec tes chiffres)
1. **Open-world** : quel détecteur et quel modèle détectent le mieux l'inconnu (AUROC, FPR@95) ? Quel écart entre les plis `random` et `family` ?
2. **Classes problématiques** : lesquelles échappent à la détection, et vers quelles classes connues sont-elles aiguillées ? Lien avec les confusions *closed-set* de l'article.
3. **Solution S1** (multi-entropie + bruit, budget constant) : effet sur la robustesse closed-set et sur la détection de l'inconnu sous entropie élevée / bruit.
4. **Solution S2** (mots + caractères + calibration) : effet sur l'AUROC du MSP, l'ECE et le F1.
5. **Deep learning** : le LSTM est-il sur-confiant ? Le détecteur d'énergie corrige-t-il ce défaut ?
6. **Limites** : classes connues découpées au niveau des logs (optimiste en closed-set) ; niveaux d'entropie issus des mêmes 440 gabarits (la robustesse mesurée est celle aux perturbations de surface) ; une seule graine par pli ; sous-échantillons par classe.

### 8.2 Pistes
- Répéter avec plusieurs graines pour les plis `random`, et tester statistiquement (Wilcoxon sur les 6 plis).
- Détecteurs supplémentaires : kNN dans l'espace d'embeddings, OpenMax, classifieur avec classe « rejet » entraînée sur des pseudo-inconnus.
- Ajouter **DistilBERT** (embeddings + énergie) au volet C.
- Évaluer sur les **logs réels** (CDS) en traitant ses classes comme inconnues ou connues selon le pli.